<div align="center">
  <img src="https://raw.githubusercontent.com/harishs-io/lateral_flowimmunoassay_quant/main/assets/pipeline_workflow.svg" width="850">
  <h1>🧬 Automated LFIA Signal Quantification</h1>
  <h3>Publication-grade bioanalytical pipeline using YOLOv8</h3>
</div>

---
**Instructions:**
1. Run the **Environment Setup** cell to load the pipeline.
2. Run the **Upload & Analyze** cell to insert your image and extract the signal.
3. Run the **Export** cell to download your data.

In [ ]:
#@title ⚙️ 1. Environment Setup { display-mode: "form" }
#@markdown Run this cell once to configure the workspace. Installation logs will be displayed below.

!git clone https://github.com/harishs-io/lateral_flowimmunoassay_quant.git
%cd lateral_flowimmunoassay_quant
!pip install ultralytics opencv-python matplotlib scipy pandas numpy

import os
import sys
sys.path.append('.')
from google.colab import files
from IPython.display import Image, display

print("\n✅ Pipeline environment successfully loaded! Move to Step 2.")

In [ ]:
#@title 🔬 2. Upload Image & Run Pipeline { display-mode: "form" }
#@markdown Adjust the detection parameters and choose whether to fine-tune the model with your image.

yolo_confidence = 0.20 #@param {type:"slider", min:0.1, max:1.0, step:0.05}
fine_tune_yolo = "No" #@param ["Yes", "No"]

import shutil
import builtins
import glob
import os

# Map the UI dropdown to the script's input prompt
auto_response = 'y' if fine_tune_yolo == "Yes" else 'n'
builtins.input = lambda prompt: auto_response

print("👇 Click 'Choose Files' below to insert your assay image:")
uploaded = files.upload()

if uploaded:
    image_name = list(uploaded.keys())[0]
    print(f"\n✅ Successfully uploaded: {image_name}")
    print("⏳ Processing image through YOLOv8 and extracting chromatogram...\n")
    
    # Recreate the folder structure the script expects
    base_dir = "colab_run"
    sub_dir = os.path.join(base_dir, "assay_01")
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)
    os.makedirs(sub_dir, exist_ok=True)
    
    # Move the uploaded image to the target directory
    image_path = os.path.join(sub_dir, image_name)
    shutil.move(image_name, image_path)
    
    try:
        from src.lfia_analyzer import run_pipeline
        
        # Run the exact pipeline script
        run_pipeline(base_dir=base_dir, dataset_dir="dataset", weights_path="weights/best.pt", conf_thresh=yolo_confidence)
        
        print("\n✅ Pipeline complete! Target localized and integrated.")
        
        # Locate and display the generated matplotlib plot
        plots = glob.glob(f"{sub_dir}/Results/**/*_Analysis_Plot.png", recursive=True)
        if plots:
            display(Image(filename=plots[0]))
        else:
            print("⚠️ Analysis finished, but the output plot could not be found.")
            
    except Exception as e:
        print(f"\n⚠️ Pipeline Error: {e}")
else:
    print("⚠️ No file uploaded. Please run the cell again.")

In [ ]:
#@title 📊 3. Export Data & Reports { display-mode: "form" }
#@markdown Zips the generated `Results/` folder (including cropped images, text reports, and plots) and downloads it to your machine.

try:
    shutil.make_archive("LFIA_Results", 'zip', "colab_run/assay_01/Results")
    print("✅ Results packaged. Triggering download for 'LFIA_Results.zip'...")
    files.download('LFIA_Results.zip')
except Exception as e:
    print("⚠️ Error: Please run Step 2 first to generate the results.")